# 🏥 스마트병동 환자안전 모니터링 — 실시간 추론 데모

**동작 방식**: 동영상 → 2초마다 프레임 추출 → Fine-tuned MobileVLM 추론 → 상태 변화 감지 → 타임라인 보고서 출력

```
00:00 환자 침대에서 휴식 중
00:37 환자 침대 이탈, 이동 중
01:22 환자 낙상 발생 → ⚠️ 간호사 호출
```

## 🔧 Cell 1: 환경 설치

In [ ]:
!pip install -q transformers==4.40.0 peft==0.9.0 bitsandbytes>=0.44.0 accelerate
!pip install -q pillow opencv-python-headless

import os, sys
if not os.path.exists('/content/MobileVLM'):
    !git clone -q https://github.com/Meituan-AutoML/MobileVLM /content/MobileVLM
sys.path.insert(0, '/content/MobileVLM')
print('✅ 환경 설치 완료')

## 📁 Cell 2: Google Drive 마운트 & 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── 경로 설정 (필요시 수정) ────────────────────────────────────
DRIVE_BASE   = '/content/drive/MyDrive/fall_dataset'
ADAPTER_DIR  = f'{DRIVE_BASE}/mobilevlm_qlora_adapter'
VIDEO_DIR    = f'{DRIVE_BASE}/demo_videos'   # ← 영상 3개 넣어두는 폴더
RESULT_DIR   = f'{DRIVE_BASE}/demo_results'

os.makedirs(RESULT_DIR, exist_ok=True)

# 영상 목록 확인
video_files = sorted([
    f for f in os.listdir(VIDEO_DIR)
    if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))
])
print(f'✅ 발견된 영상: {len(video_files)}개')
for v in video_files:
    print(f'  - {v}')

## 🤖 Cell 3: Fine-tuned 모델 로드

In [ ]:
import torch
from transformers import BitsAndBytesConfig
from peft import PeftModel
from PIL import Image
from mobilevlm.model.mobilevlm import load_pretrained_model
from mobilevlm.utils import disable_torch_init
from mobilevlm.conversation import conv_templates

disable_torch_init()

MODEL_NAME = 'mtgv/MobileVLM_V2-1.7B'
print('모델 로드 중...')

tokenizer, model_base, image_processor, _ = load_pretrained_model(
    model_path=MODEL_NAME,
    model_base=None,
    model_name='MobileVLM_V2-1.7B',
    load_4bit=True,
)

# Fine-tuned LoRA adapter 적용
model = PeftModel.from_pretrained(model_base, ADAPTER_DIR)
model.eval()
print('✅ Fine-tuned 모델 로드 완료')

## ⚙️ Cell 4: 추론 함수 & 설정

In [ ]:
from mobilevlm.utils import process_images, tokenizer_image_token
from mobilevlm.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN

# ── 설정 ──────────────────────────────────────────────────────
INTERVAL_SEC  = 2        # 추론 간격 (초)
PROMPT        = '현재 환자의 상태를 보고하세요.'
ALERT_KEYWORD = '낙상'    # 이 단어 감지 시 즉시 알림

def infer_frame(pil_image):
    """PIL 이미지 1장 → 추론 결과 문자열 반환"""
    image_tensor = process_images([pil_image], image_processor, model.config)[0]
    image_tensor = image_tensor.to(model.device, dtype=torch.float16)

    full_prompt = DEFAULT_IMAGE_TOKEN + '\n' + PROMPT
    conv = conv_templates['llava_v1'].copy()
    conv.append_message(conv.roles[0], full_prompt)
    conv.append_message(conv.roles[1], None)
    prompt_str = conv.get_prompt()

    input_ids = tokenizer_image_token(
        prompt_str, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt'
    ).unsqueeze(0).to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            input_ids,
            images=image_tensor.unsqueeze(0),
            do_sample=False,
            max_new_tokens=64,
            use_cache=True,
        )
    return tokenizer.decode(
        output_ids[0, input_ids.shape[1]:], skip_special_tokens=True
    ).strip()

def format_time(seconds):
    """초 → MM:SS 형식"""
    return f'{int(seconds)//60:02d}:{int(seconds)%60:02d}'

print(f'✅ 설정 완료')
print(f'   추론 간격: {INTERVAL_SEC}초')
print(f'   프롬프트:  {PROMPT}')
print(f'   경보 키워드: {ALERT_KEYWORD}')

## 🎬 Cell 5: 영상 선택 & 실시간 추론 실행

In [ ]:
import cv2
import time

# ── 처리할 영상 선택 (0, 1, 2 중 선택) ────────────────────────
VIDEO_INDEX = 0
video_path  = os.path.join(VIDEO_DIR, video_files[VIDEO_INDEX])
print(f'영상: {video_files[VIDEO_INDEX]}')

# ── 영상 정보 ──────────────────────────────────────────────────
cap = cv2.VideoCapture(video_path)
fps         = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration    = total_frames / fps
frame_step  = int(fps * INTERVAL_SEC)   # N초마다 1프레임

print(f'FPS: {fps:.1f} | 총 프레임: {total_frames} | 영상 길이: {format_time(duration)}')
print(f'분석 프레임 수: {total_frames // frame_step}장 ({INTERVAL_SEC}초 간격)')
print()
print('=' * 60)
print('📋 환자 행동 보고서')
print('=' * 60)

# ── 추론 루프 ──────────────────────────────────────────────────
report      = []       # (timestamp, status) 리스트
prev_status = None
frame_idx   = 0
alert_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # INTERVAL_SEC마다 1번만 추론
    if frame_idx % frame_step != 0:
        frame_idx += 1
        continue

    timestamp  = frame_idx / fps
    time_label = format_time(timestamp)

    # BGR(OpenCV) → RGB → PIL
    pil_img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    t0     = time.time()
    status = infer_frame(pil_img)
    elapsed = time.time() - t0

    # 상태 변화 감지
    changed = (status != prev_status)

    if changed or prev_status is None:
        alert = ''
        if ALERT_KEYWORD in status:
            alert = '  ⚠️  → 간호사 호출!'
            alert_count += 1
        print(f'{time_label}  {status}{alert}')
        report.append({'time': time_label, 'status': status, 'alert': bool(alert)})
        prev_status = status
    else:
        # 변화 없음 → 점만 찍어서 진행 상황 표시
        print(f'{time_label}  (상태 유지: {status[:20]}...) [{elapsed:.1f}s]')

    frame_idx += 1

cap.release()

print()
print('=' * 60)
print(f'✅ 분석 완료 | 상태 변화: {len(report)}회 | 낙상 경보: {alert_count}회')

## 📝 Cell 6: 최종 보고서 출력 & 저장

In [ ]:
from datetime import datetime

print('\n' + '=' * 60)
print('📋 최종 환자 행동 타임라인 보고서')
print('=' * 60)
for entry in report:
    alert_str = '  ⚠️ 낙상 경보!' if entry['alert'] else ''
    print(f"{entry['time']}  {entry['status']}{alert_str}")
print('=' * 60)

# ── 텍스트 파일 저장 ──────────────────────────────────────────
video_name   = os.path.splitext(video_files[VIDEO_INDEX])[0]
report_path  = f'{RESULT_DIR}/{video_name}_report.txt'

with open(report_path, 'w', encoding='utf-8') as f:
    f.write(f'환자 행동 모니터링 보고서\n')
    f.write(f'생성일시: {datetime.now().strftime("%Y-%m-%d %H:%M")}\n')
    f.write(f'영상: {video_files[VIDEO_INDEX]}\n')
    f.write(f'모델: Fine-tuned MobileVLM V2 1.7B (QLoRA)\n')
    f.write('=' * 40 + '\n')
    for entry in report:
        alert_str = '  ⚠️ 낙상 경보!' if entry['alert'] else ''
        f.write(f"{entry['time']}  {entry['status']}{alert_str}\n")
    f.write('=' * 40 + '\n')
    f.write(f'총 상태 변화: {len(report)}회\n')
    f.write(f'낙상 경보 발생: {alert_count}회\n')

print(f'\n✅ 보고서 저장: {report_path}')

# 로컬 다운로드
from google.colab import files
files.download(report_path)

## 🔁 Cell 7: 전체 영상 일괄 처리 (선택)
Cell 5~6을 모든 영상에 대해 자동 반복

In [ ]:
for vid_idx, vid_file in enumerate(video_files):
    video_path = os.path.join(VIDEO_DIR, vid_file)

    print(f'\n{'='*60}')
    print(f'[{vid_idx+1}/{len(video_files)}] {vid_file}')
    print(f'{'='*60}')

    cap         = cv2.VideoCapture(video_path)
    fps         = cap.get(cv2.CAP_PROP_FPS)
    frame_step  = max(1, int(fps * INTERVAL_SEC))
    report_b    = []
    prev_status = None
    frame_idx   = 0
    alert_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_step != 0:
            frame_idx += 1
            continue

        timestamp  = frame_idx / fps
        time_label = format_time(timestamp)
        pil_img    = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        status     = infer_frame(pil_img)

        if status != prev_status or prev_status is None:
            has_alert = ALERT_KEYWORD in status
            if has_alert:
                alert_count += 1
            alert_str = '  ⚠️ 간호사 호출!' if has_alert else ''
            print(f'{time_label}  {status}{alert_str}')
            report_b.append({'time': time_label, 'status': status, 'alert': has_alert})
            prev_status = status
        frame_idx += 1

    cap.release()

    # 보고서 저장
    vname = os.path.splitext(vid_file)[0]
    rpath = f'{RESULT_DIR}/{vname}_report.txt'
    with open(rpath, 'w', encoding='utf-8') as f:
        f.write(f'환자 행동 모니터링 보고서\n생성일시: {datetime.now().strftime("%Y-%m-%d %H:%M")}\n')
        f.write(f'영상: {vid_file}\n' + '='*40 + '\n')
        for e in report_b:
            f.write(f"{e['time']}  {e['status']}{'  ⚠️' if e['alert'] else ''}\n")
        f.write(f'{'='*40}\n총 변화: {len(report_b)}회 | 낙상 경보: {alert_count}회\n')

    print(f'✅ 저장 완료: {rpath}')

print('\n🎉 전체 영상 처리 완료!')